In [1]:
import pandas as pd

df = pd.read_csv("final_military_cleaned.csv")
print(df.shape)
df.head()

(145, 35)


,Country,total_population,active_personnel,reserve_personnel,paramilitary,total_military_aircraft,fighter_aircraft,attack_aircraft,transport_aircraft,trainer_aircraft,...,corvettes,coastal_patrol_craft,mine_warfare_craft,defense_budget_usd,purchasing_power_parity_usd,foreign_exchange_and_gold_reserves_usd,total_serviceable_airports,major_ports_and_terminals,railway_coverage_km,roadway_coverage_km
0,United States,341963408,1328000,799500,0,13043,1790,889,918,2647,...,26,0,8,895000000000,24662000000000,773426000000,15873,666,293564,6586610
1,Russia,140820810,1320000,2000000,250000,4292,833,689,456,611,...,83,123,47,126000000000,5816000000000,597217000000,904,67,85494,1283387
2,China,1415043270,2035000,510000,625000,3309,1212,371,289,402,...,72,150,36,266850000000,31227000000000,3450000000000,531,66,150000,5200000
3,India,1409128296,1455550,1155000,2527000,2229,513,130,270,351,...,18,135,0,75000000000,13104000000000,627793000000,311,56,65554,6371847
4,South Korea,52081799,600000,3100000,120000,1592,315,98,41,318,...,5,35,14,46300000000,2615000000000,420930000000,89,15,3979,100428


In [2]:
print(df.columns.tolist())

['Country', 'total_population', 'active_personnel', 'reserve_personnel', 'paramilitary', 'total_military_aircraft', 'fighter_aircraft', 'attack_aircraft', 'transport_aircraft', 'trainer_aircraft', 'special_mission_aircraft', 'tanker_aircraft', 'total_military_helicopters', 'attack_helicopters', 'tanks', 'armored_fighting_vehicles', 'self_propelled_artillery', 'towed_artillery', 'rocket_projectors', 'total_naval_fleet', 'aircraft_carriers', 'helicopter_carriers', 'submarines', 'destroyers', 'frigates', 'corvettes', 'coastal_patrol_craft', 'mine_warfare_craft', 'defense_budget_usd', 'purchasing_power_parity_usd', 'foreign_exchange_and_gold_reserves_usd', 'total_serviceable_airports', 'major_ports_and_terminals', 'railway_coverage_km', 'roadway_coverage_km']


In [3]:
df["total_air_assets"] = (
    df["total_military_aircraft"] +
    df["total_military_helicopters"]
)

df["total_land_assets"] = (
    df["tanks"] +
    df["armored_fighting_vehicles"] +
    df["self_propelled_artillery"] +
    df["towed_artillery"] +
    df["rocket_projectors"]
)

df["total_naval_assets"] = df["total_naval_fleet"]

df["total_assets"] = (
    df["total_air_assets"] +
    df["total_land_assets"] +
    df["total_naval_assets"]
)

In [4]:
#KPI 1
df["assets_per_capita"] = (df["total_assets"] / df["total_population"]) * 1_000_000

In [5]:
#KPI 2
df["budget_to_gdp_ratio"] = (
    df["defense_budget_usd"] / df["purchasing_power_parity_usd"]
)

In [6]:
#KPI 3
df["power_score"] = (
    df["total_assets"] * 0.5 +
    df["active_personnel"] * 0.3 +
    df["defense_budget_usd"] * 0.2
)

In [7]:
#generating rank from power score
df["computed_rank"] = df["power_score"].rank(ascending=False, method="dense")

In [8]:
#rank gap
df["power_index_rank_gap"] = df["computed_rank"] - df["computed_rank"].min()

In [9]:
#mapping
region_map = {
    "United States": ("North America", "NATO"),
    "Canada": ("North America", "NATO"),
    "Germany": ("Europe", "NATO"),
    "France": ("Europe", "NATO"),
    "United Kingdom": ("Europe", "NATO"),
    "India": ("Asia", "Non-NATO"),
    "China": ("Asia", "Non-NATO"),
    "Russia": ("Europe/Asia", "Non-NATO"),
    "Japan": ("Asia", "Non-NATO"),
    "South Korea": ("Asia", "Non-NATO")
}

In [10]:
#metadata
df["region"] = df["Country"].map(lambda x: region_map.get(x, ("Other", "Non-NATO"))[0])
df["alliance"] = df["Country"].map(lambda x: region_map.get(x, ("Other", "Non-NATO"))[1])

In [11]:
wide_df = df.copy()

In [12]:
kpi_cols = [
    "assets_per_capita",
    "budget_to_gdp_ratio",
    "power_index_rank_gap"
]

long_df = pd.melt(
    df,
    id_vars=["Country", "region", "alliance"],
    value_vars=kpi_cols,
    var_name="KPI",
    value_name="Value"
)

In [13]:
with pd.ExcelWriter("military_final.xlsx", engine="openpyxl") as writer:
    wide_df.to_excel(writer, sheet_name="Wide_Format", index=False)
    long_df.to_excel(writer, sheet_name="Long_Format", index=False)

print("military_final.xlsx created successfully")

military_final.xlsx created successfully


In [14]:
df[["Country", "total_assets", "total_population", "assets_per_capita"]].head(10)

,Country,total_assets,total_population,assets_per_capita
0,United States,418453,341963408,1223.677710
1,Russia,160317,140820810,1138.446796
2,China,163033,1415043270,115.214145
3,India,160555,1409128296,113.939235
4,South Korea,71838,52081799,1379.330234
5,United Kingdom,39659,68459055,579.309779
6,France,112805,68374591,1649.808772
7,Japan,35366,123201945,287.057156
8,Turkiye,68225,84119531,811.048269
9,Italy,75190,60964931,1233.331995


In [15]:
df["total_naval_assets"] = df["total_naval_fleet"]

In [16]:
df["assets_per_capita"] = (df["total_assets"] / df["total_population"]) * 1_000_000